# Preparación de conformers 3D de AqSolDB (T9.0)

Genera varios conformers por molécula de las particiones oficiales de AqSolDB, optimiza con un único campo de fuerza, selecciona el conformer convergido de menor energía finita y persiste exactamente un conformer por molécula válida.

Salidas:
- data/conformers/aqsoldb/conformers.pkl
- data/conformers/aqsoldb/manifest.json
- data/conformers/aqsoldb/excluded.csv

No construye grafos, no normaliza features y no entrena modelos.

In [ ]:
import hashlib
import json
import pickle
import platform
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from rdkit import Chem, rdBase
from tdc.benchmark_group import admet_group
from tqdm.auto import tqdm

from src.features.conformers import (
    generate_conformers,
    optimize_conformers,
    get_lowest_energy_conformer,
    keep_only_conformer,
    validate_conformer_geometry,
)

In [ ]:
def find_repo_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "docs" / "gnn" / "spec.md").is_file():
            return candidate
    raise RuntimeError("Could not locate repository root; run from the repository.")

REPO_ROOT = find_repo_root()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"
OUTPUT_DIR = REPO_ROOT / "data" / "conformers" / "aqsoldb"

# Fixed, recorded parameters. One thread keeps embedding/optimization deterministic.
BASE_SEED = 42
N_CONFORMERS = 10
PRUNE_RMS_THRESH = 0.5
NUM_THREADS = 1
FORCE_FIELD = "MMFF"
MAX_ITERS = 200
MIN_PAIRWISE_DISTANCE_ANGSTROM = 0.05

if FORCE_FIELD not in {"MMFF", "UFF"}:
    raise ValueError("FORCE_FIELD must be exactly 'MMFF' or 'UFF'.")

print("Repository:", REPO_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Output:", OUTPUT_DIR)
print({
    "base_seed": BASE_SEED,
    "n_conformers": N_CONFORMERS,
    "prune_rms_thresh": PRUNE_RMS_THRESH,
    "num_threads": NUM_THREADS,
    "force_field": FORCE_FIELD,
    "max_iters": MAX_ITERS,
    "min_pairwise_distance_angstrom": MIN_PAIRWISE_DISTANCE_ANGSTROM,
})

In [ ]:
# Load the official TDC partitions without modifying their rows.
group = admet_group(path=str(RAW_DATA_DIR))
benchmark = group.get("Solubility_AqSolDB")

required_columns = {"Drug_ID", "Drug", "Y"}
partition_frames = []
partition_sizes = {}
for split_name in ("train_val", "test"):
    frame = benchmark[split_name].copy()
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"{split_name} is missing columns: {sorted(missing)}")
    partition_sizes[split_name] = int(len(frame))
    partition_frames.append(frame.assign(split=split_name))

benchmark_records = pd.concat(partition_frames, ignore_index=True).rename(
    columns={"Drug_ID": "mol_id", "Drug": "smiles", "Y": "y"}
)
if benchmark_records.empty:
    raise ValueError("Solubility_AqSolDB contains no records.")

print("Input records:", len(benchmark_records))
print("Partition sizes:", partition_sizes)

In [ ]:
def seed_for_molecule(mol_id, row_index):
    # Stable per-molecule seed; row index is only a fallback for missing IDs.
    identity = f"id:{mol_id}" if pd.notna(mol_id) else f"row:{row_index}"
    digest = hashlib.sha256(f"{BASE_SEED}:{identity}".encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big") % (2**31 - 1) + 1


def prepare_record(row, row_index):
    mol_id, smiles, target, split = (
        row.get("mol_id"), row.get("smiles"), row.get("y"), row.get("split")
    )
    seed = seed_for_molecule(mol_id, row_index)
    stage = "input_validation"
    try:
        if pd.isna(mol_id):
            raise ValueError("mol_id is missing.")
        if pd.isna(smiles) or not str(smiles).strip():
            raise ValueError("SMILES is missing.")
        if pd.isna(target) or not np.isfinite(float(target)):
            raise ValueError("Target y is missing or non-finite.")
        if split not in {"train_val", "test"}:
            raise ValueError(f"Unexpected split: {split!r}.")

        stage = "smiles_parsing"
        source_mol = Chem.MolFromSmiles(str(smiles))
        if source_mol is None:
            raise ValueError("RDKit MolFromSmiles returned None.")

        stage = "conformer_embedding"
        mol, conf_ids = generate_conformers(
            source_mol, n_conformers=N_CONFORMERS, random_seed=seed,
            prune_rms_thresh=PRUNE_RMS_THRESH, num_threads=NUM_THREADS,
        )

        stage = "force_field_optimization"
        results = optimize_conformers(
            mol, force_field=FORCE_FIELD, max_iters=MAX_ITERS,
            num_threads=NUM_THREADS,
        )
        if len(conf_ids) != len(results):
            raise ValueError("Conformer IDs and optimization results are misaligned.")

        stage = "conformer_selection"
        best_conf_id = get_lowest_energy_conformer(conf_ids, results)
        best_position = conf_ids.index(best_conf_id)
        best_energy = float(results[best_position][1])

        stage = "geometry_validation"
        validate_conformer_geometry(
            mol, best_conf_id,
            min_pairwise_distance=MIN_PAIRWISE_DISTANCE_ANGSTROM,
        )

        stage = "selected_conformer_persistence"
        selected_mol = keep_only_conformer(mol, best_conf_id)
        if selected_mol.GetNumConformers() != 1:
            raise RuntimeError("Expected exactly one persisted conformer.")
        if selected_mol.GetConformer().GetId() != best_conf_id:
            raise RuntimeError("The selected RDKit conf_id was not preserved.")

        return {
            "mol_id": mol_id, "smiles": smiles, "y": target, "split": split,
            "seed": int(seed), "mol": selected_mol, "conf_id": int(best_conf_id),
            "force_field": FORCE_FIELD, "energy": best_energy,
        }, None
    except Exception as exc:
        return None, {
            "mol_id": mol_id, "smiles": smiles, "y": target, "split": split,
            "seed": int(seed), "stage": stage,
            "error_type": type(exc).__name__, "reason": str(exc),
        }

In [ ]:
records, excluded = [], []
rows = benchmark_records.to_dict(orient="records")
for row_index, row in enumerate(tqdm(rows, total=len(rows), desc="AqSolDB conformers")):
    record, failure = prepare_record(row, row_index)
    if record is not None:
        records.append(record)
    else:
        excluded.append(failure)

if len(records) + len(excluded) != len(benchmark_records):
    raise RuntimeError("Some input rows were neither persisted nor recorded as excluded.")
if not records:
    raise RuntimeError("No valid conformers were generated; inspect exclusions.")

print("Valid:", len(records), "Excluded:", len(excluded))

In [ ]:
valid_by_split = Counter(record["split"] for record in records)
excluded_by_split = Counter(record["split"] for record in excluded)
excluded_by_stage = Counter(record["stage"] for record in excluded)

manifest = {
    "format_version": 1,
    "dataset": "Solubility_AqSolDB",
    "source": "TDC official train_val and test partitions",
    "input_records": int(len(benchmark_records)),
    "valid_records": int(len(records)),
    "excluded_records": int(len(excluded)),
    "input_records_by_split": {str(k): int(v) for k, v in benchmark_records["split"].value_counts().items()},
    "valid_records_by_split": {str(k): int(v) for k, v in valid_by_split.items()},
    "excluded_records_by_split": {str(k): int(v) for k, v in excluded_by_split.items()},
    "excluded_records_by_stage": {str(k): int(v) for k, v in excluded_by_stage.items()},
    "seed": {
        "base_seed": BASE_SEED,
        "strategy": "SHA256(base_seed + stable mol_id; row index fallback if missing)",
        "per_record_field": "seed",
    },
    "embedding": {
        "method": "ETKDGv3", "n_conformers": N_CONFORMERS,
        "prune_rms_thresh_angstrom": PRUNE_RMS_THRESH, "num_threads": NUM_THREADS,
    },
    "optimization": {
        "force_field": FORCE_FIELD, "max_iters": MAX_ITERS,
        "num_threads": NUM_THREADS, "eligible_status": 0,
        "eligible_energy": "finite", "fallback_force_field": None,
    },
    "geometry_validation": {
        "coordinates_finite": True,
        "minimum_pairwise_distance_angstrom": MIN_PAIRWISE_DISTANCE_ANGSTROM,
        "planar_geometries_allowed": True,
    },
    "hydrogens": {"policy": "explicit", "removed": False},
    "versions": {
        "python": platform.python_version(), "numpy": np.__version__,
        "pandas": pd.__version__, "rdkit": rdBase.rdkitVersion,
    },
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
conformers_path = OUTPUT_DIR / "conformers.pkl"
manifest_path = OUTPUT_DIR / "manifest.json"
excluded_path = OUTPUT_DIR / "excluded.csv"

with conformers_path.open("wb") as handle:
    pickle.dump(records, handle, protocol=pickle.HIGHEST_PROTOCOL)
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
excluded_columns = ["mol_id", "smiles", "y", "split", "seed", "stage", "error_type", "reason"]
pd.DataFrame(excluded, columns=excluded_columns).to_csv(excluded_path, index=False)

print("Saved:", conformers_path)
print("Saved:", manifest_path)
print("Saved:", excluded_path)

In [ ]:
# Read-back checks for the exact artifact contract consumed by T9.1.
with conformers_path.open("rb") as handle:
    persisted = pickle.load(handle)

required_fields = {"mol_id", "smiles", "y", "split", "seed", "mol", "conf_id"}
assert len(persisted) == len(records)
assert all(required_fields <= set(record) for record in persisted)
assert all(record["mol"].GetNumConformers() == 1 for record in persisted)
assert all(record["mol"].GetConformer().GetId() == record["conf_id"] for record in persisted)
assert all(any(atom.GetAtomicNum() == 1 for atom in record["mol"].GetAtoms()) for record in persisted)
assert len(persisted) + len(excluded) == len(benchmark_records)

print("Persistence checks passed.")
print("Records ready for T9.1:", len(persisted))
print("Excluded records:", len(excluded))